# Compare

In this notebook, we compare the performance of different causal inference methods using various metrics. We load multiple datasets generated by different methods and evaluate how well each method identifies causal relationships compared to the ground truth.

We employ metrics such as accuracy, balanced accuracy, precision, recall, and F1-score to assess the performance of each method. Additionally, we aggregate the results into a comprehensive table for easy comparison and analysis.


In [21]:
import pickle 
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score, roc_auc_score

In [22]:
with open('data/causal_dfs_dream3.pkl', 'rb') as f:
    causal_dfs_var, causal_dfs_varlingam, causal_dfs_pcmci,causal_dfs_mvgc, causal_dfs_pcmci_gpdc, causal_dfs_granger, causal_dfs_dynotears, causal_dfs_d2c, true_causal_dfs = pickle.load(f)
    

In [23]:

# causal_dfs_varlingam = pd.concat(causal_dfs_varlingam.values()).reset_index(drop=True)
causal_dfs_var = pd.concat(causal_dfs_var.values()).reset_index(drop=True)
causal_dfs_pcmci = pd.concat(causal_dfs_pcmci.values()).reset_index(drop=True)
# causal_dfs_mvgc = pd.concat(causal_dfs_mvgc.values()).reset_index(drop=True)
# causal_dfs_pcmci_gpdc = pd.concat(causal_dfs_pcmci_gpdc.values()).reset_index(drop=True)
causal_dfs_granger = pd.concat(causal_dfs_granger.values()).reset_index(drop=True)
causal_dfs_dynotears = pd.concat(causal_dfs_dynotears.values()).reset_index(drop=True)
causal_dfs_d2c = pd.concat(causal_dfs_d2c.values()).reset_index(drop=True)
true_causal_dfs = pd.concat(true_causal_dfs).reset_index(drop=True)

In [24]:
causal_dfs_d2c

,from,to,effect,p_value,probability,is_causal
0,50,0,None,0.362,0.638,1
1,50,1,None,0.902,0.098,0
2,50,2,None,0.888,0.112,0
3,50,3,None,0.888,0.112,0
4,50,4,None,0.834,0.166,0
...,...,...,...,...,...,...
37495,199,45,None,0.914,0.086,0
37496,199,46,None,0.958,0.042,0
37497,199,47,None,0.862,0.138,0
37498,199,48,None,0.854,0.146,0


In [5]:
# We perform a sanity check to ensure that the causal_dfs are the same across all methods

assert (causal_dfs_varlingam['from'] == causal_dfs_var['from']).all()
assert (causal_dfs_varlingam['from'] == causal_dfs_pcmci['from']).all()
# assert (causal_dfs_varlingam['from'] == causal_dfs_mvgc['from']).all()
# assert (causal_dfs_varlingam['from'] == causal_dfs_pcmci_gpdc['from']).all()
assert (causal_dfs_varlingam['from'] == causal_dfs_granger['from']).all()
assert (causal_dfs_varlingam['from'] == causal_dfs_dynotears['from']).all()
assert (causal_dfs_varlingam['from'] == causal_dfs_d2c['from']).all()
assert (causal_dfs_varlingam['from'] == true_causal_dfs['from']).all()

assert (causal_dfs_varlingam['to'] == causal_dfs_var['to']).all()
assert (causal_dfs_varlingam['to'] == causal_dfs_pcmci['to']).all()
assert (causal_dfs_varlingam['to'] == causal_dfs_mvgc['to']).all()
assert (causal_dfs_varlingam['to'] == causal_dfs_pcmci_gpdc['to']).all()
assert (causal_dfs_varlingam['to'] == causal_dfs_granger['to']).all()
assert (causal_dfs_varlingam['to'] == causal_dfs_dynotears['to']).all()
assert (causal_dfs_varlingam['to'] == causal_dfs_d2c['to']).all()
assert (causal_dfs_varlingam['to'] == true_causal_dfs['to']).all()


In [25]:
y_true = true_causal_dfs['is_causal'].astype(int)
y_pred_var = causal_dfs_var['is_causal'].astype(int)
y_pred_pcmci = causal_dfs_pcmci['is_causal'].astype(int)
# y_pred_mvgc = causal_dfs_mvgc['is_causal'].astype(int)
# y_pred_pcmci_gpdc = causal_dfs_pcmci_gpdc['is_causal'].astype(int)
y_pred_granger = causal_dfs_granger['is_causal'].astype(int)
y_pred_dynotears = causal_dfs_dynotears['is_causal'].astype(int)
y_pred_d2c = causal_dfs_d2c['is_causal'].astype(int)
y_proba_d2c = causal_dfs_d2c['probability'].astype(float)
# y_pred_varlingam = causal_dfs_varlingam['is_causal'].astype(int)
# y_proba_varlingam = causal_dfs_varlingam['probability'].astype(float)

In [7]:
concat_predictions = pd.concat([y_pred_var, 
                                y_pred_varlingam,
                                y_pred_pcmci, 
                                y_pred_mvgc,
                                y_pred_pcmci_gpdc,
                                y_pred_granger, 
                                y_pred_dynotears,
                                y_pred_d2c, 
                                y_true], axis=1)
concat_predictions.columns = ['VAR', 'VarLingam', 'PCMCI', 'MVGC', 'PCMCI-GPDC', 'Granger', 'Dynotears', 'D2C', 'True']

In [26]:
scores = pd.DataFrame(
    columns=[
        "Method",
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC",
        "Total",
        "Positive",
    ]
)

scores.loc[len(scores)] = [
    "VAR",
    accuracy_score(y_true, y_pred_var),
    balanced_accuracy_score(y_true, y_pred_var),
    precision_score(y_true, y_pred_var, zero_division=np.nan),
    recall_score(y_true, y_pred_var, zero_division=np.nan),
    f1_score(y_true, y_pred_var, zero_division=np.nan),
    np.nan,
    len(y_true),
    y_true.sum(),
]

scores.loc[len(scores)] = [
    "PCMCI",
    accuracy_score(y_true, y_pred_pcmci),
    balanced_accuracy_score(y_true, y_pred_pcmci),
    precision_score(y_true, y_pred_pcmci, zero_division=np.nan),
    recall_score(y_true, y_pred_pcmci, zero_division=np.nan),
    f1_score(y_true, y_pred_pcmci, zero_division=np.nan),
    np.nan,
    len(y_true),
    y_true.sum(),
]

# scores.loc[len(scores)] = [
#     "MVGC",
#     accuracy_score(y_true, y_pred_mvgc),
#     balanced_accuracy_score(y_true, y_pred_mvgc),
#     precision_score(y_true, y_pred_mvgc, zero_division=np.nan),
#     recall_score(y_true, y_pred_mvgc, zero_division=np.nan),
#     f1_score(y_true, y_pred_mvgc, zero_division=np.nan),
#     np.nan,
#     len(y_true),
#     y_true.sum(),
# ]

# scores.loc[len(scores)] = [
#     "PCMCI - GPDC ",
#     accuracy_score(y_true, y_pred_pcmci_gpdc),
#     balanced_accuracy_score(y_true, y_pred_pcmci_gpdc),
#     precision_score(y_true, y_pred_pcmci_gpdc, zero_division=np.nan),
#     recall_score(y_true, y_pred_pcmci_gpdc, zero_division=np.nan),
#     f1_score(y_true, y_pred_pcmci_gpdc, zero_division=np.nan),
#     np.nan,
#     len(y_true),
#     y_true.sum(),
# ]


scores.loc[len(scores)] = [
    "Granger",
    accuracy_score(y_true, y_pred_granger),
    balanced_accuracy_score(y_true, y_pred_granger),
    precision_score(y_true, y_pred_granger, zero_division=np.nan),
    recall_score(y_true, y_pred_granger, zero_division=np.nan),
    f1_score(y_true, y_pred_granger, zero_division=np.nan),
    np.nan,
    len(y_true),
    y_true.sum(),
]

scores.loc[len(scores)] = [
    "Dynotears",
    accuracy_score(y_true, y_pred_dynotears),
    balanced_accuracy_score(y_true, y_pred_dynotears),
    precision_score(y_true, y_pred_dynotears, zero_division=np.nan),
    recall_score(y_true, y_pred_dynotears, zero_division=np.nan),
    f1_score(y_true, y_pred_dynotears, zero_division=np.nan),
    np.nan,
    len(y_true),
    y_true.sum(),
]

# scores.loc[len(scores)] = [
#     "VarLingam",
#     accuracy_score(y_true, y_pred_varlingam),
#     balanced_accuracy_score(y_true, y_pred_varlingam),
#     precision_score(y_true, y_pred_varlingam, zero_division=np.nan),
#     recall_score(y_true, y_pred_varlingam, zero_division=np.nan),
#     f1_score(y_true, y_pred_varlingam, zero_division=np.nan),
#     roc_auc_score(y_true, y_proba_varlingam),
#     len(y_true),
#     y_true.sum(),
# ]

scores.loc[len(scores)] = [
    "D2C",
    accuracy_score(y_true, y_pred_d2c),
    balanced_accuracy_score(y_true, y_pred_d2c),
    precision_score(y_true, y_pred_d2c, zero_division=np.nan),
    recall_score(y_true, y_pred_d2c, zero_division=np.nan),
    f1_score(y_true, y_pred_d2c, zero_division=np.nan),
    roc_auc_score(y_true, y_proba_d2c),
    len(y_true),
    y_true.sum(),
]

In [27]:
scores

,Method,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC AUC,Total,Positive
0,VAR,0.965227,0.512043,0.055160,0.038557,0.045388,NaN,37500,804
1,PCMCI,0.961547,0.672570,0.241491,0.370647,0.292444,NaN,37500,804
2,Granger,0.843413,0.466226,0.011188,0.072139,0.019372,NaN,37500,804
3,Dynotears,0.960640,0.523082,0.068123,0.065920,0.067004,NaN,37500,804
4,D2C,0.978000,0.656646,0.480447,0.320896,0.384787,0.750941,37500,804


# Conclusion

The results presented in this notebook indicate varying performances of different causal inference methods when applied to a limited training dataset. It's important to note that these methods typically require larger and more diverse datasets to achieve reliable and accurate results, particularly for complex tasks like causal inference. 

For more detailed and complete results, please refer to the `submission` branch or further experimentation with broader datasets.


In [12]:
# %% [markdown]
# # Compare
# 
# In this notebook, we compare the performance of different causal inference methods using various metrics. We load multiple datasets generated by different methods and evaluate how well each method identifies causal relationships compared to the ground truth.
# 
# We employ metrics such as accuracy, balanced accuracy, precision, recall, and F1-score to assess the performance of each method. Instead of a single aggregate score, we will analyze the distribution of these metrics across all datasets to understand the robustness and consistency of each method.
# 

# %%
import pickle 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score


# %%
# Define methods and their corresponding flat data dictionaries
methods = {
    'VAR': causal_dfs_var,
    'VarLingam': causal_dfs_varlingam,
    'PCMCI': causal_dfs_pcmci,
    'MVGC': causal_dfs_mvgc,
    'PCMCI-GPDC': causal_dfs_pcmci_gpdc,
    'Granger': causal_dfs_granger,
    'Dynotears': causal_dfs_dynotears,
    'D2C': causal_dfs_d2c
}

# Flatten the ground truth dictionary into a single list of DataFrames.
# The order will naturally match the integer keys of the prediction dictionaries.
all_true_dfs = true_causal_dfs

# Create a corresponding list of names for better labeling
dataset_names = [f"{data_type}_{i}" for data_type, dfs_list in true_causal_dfs.items() for i in range(len(dfs_list))]


# %%
# Calculate metrics for each dataset instance and each method
results_list = []

# Iterate through each dataset instance using its index 'i'
for i, true_df in enumerate(all_true_dfs):
    dataset_name = dataset_names[i]
    y_true = true_df['is_causal'].astype(int)
    
    # We need true positive samples to calculate meaningful recall and F1 scores.
    if y_true.sum() == 0:
        print(f"Skipping dataset {dataset_name} as it has no true positive causal links.")
        continue
        
    # For each method, get the predictions for this specific dataset instance 'i'
    for method_name, pred_dfs_dict in methods.items():
        # Check if the method has results for this specific instance key
        if i not in pred_dfs_dict:
            print(f"Warning: Missing data for method '{method_name}' on dataset {dataset_name}. Skipping.")
            continue

        pred_df = pred_dfs_dict[i]
        y_pred = pred_df['is_causal'].astype(int)

        # Sanity check to ensure we are comparing the same set of possible edges
        assert (true_df['from'].values == pred_df['from'].values).all() and (true_df['to'].values == pred_df['to'].values).all(), \
            f"Mismatch in edge definitions for method '{method_name}' on dataset {dataset_name}."
        
        # Calculate metrics, using zero_division=0.0 to handle cases with no predicted positives
        results_list.append({
            'Dataset': dataset_name,
            'Method': method_name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0.0),
            'Recall': recall_score(y_true, y_pred, zero_division=0.0),
            'F1': f1_score(y_true, y_pred, zero_division=0.0),
        })

# Convert the list of results into a DataFrame for analysis and plotting
results_df = pd.DataFrame(results_list)

# %%
# Check the head of the resulting dataframe to ensure it's populated
print("--- Results DataFrame Head ---")
if not results_df.empty:
    print(results_df.head())
else:
    print("DataFrame is empty. Please check data loading and processing steps.")
print("----------------------------")


# %% [markdown]
# ## Performance Distribution
# 
# To better understand the performance and robustness of each method across different datasets, we can visualize the distribution of their scores using boxplots. Each boxplot shows the median, quartiles, and outliers for a specific metric (e.g., F1-score) for each causal inference method. This highlights how consistently each method performs when faced with data from different generating processes.

# %%
# Set plot style
sns.set_style("whitegrid")

# Create boxplots for the key metrics
metrics_to_plot = ['F1', 'Balanced Accuracy', 'Precision', 'Recall']

# Determine the layout for subplots for a more compact visualization
n_metrics = len(metrics_to_plot)
n_cols = 2
n_rows = (n_metrics + n_cols - 1) // n_cols # Calculate rows needed

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 6 * n_rows))
axes = axes.flatten() # Flatten the axes array for easy iteration

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    sns.boxplot(x='Method', y=metric, data=results_df, ax=ax, palette="Set2")
    ax.set_title(f'Distribution of {metric} Scores', fontsize=14)
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('')
    ax.set_ylabel(metric, fontsize=12)

# Hide any unused subplots if the number of metrics is odd
for i in range(n_metrics, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout(pad=3.0)
plt.show()

# %% [markdown]
# # Conclusion
# 
# The boxplots above illustrate the performance distribution of each causal inference method across various synthetic datasets. This provides a more nuanced view than a single aggregate score.
# 
# -   **F1-Score and Balanced Accuracy:** These plots show the trade-off between precision and recall and the performance on imbalanced data, respectively. We can observe which methods consistently achieve higher scores and which have more variance in their performance, indicating sensitivity to the underlying data generating process.
# -   **Precision and Recall:** These plots reveal the specific strengths and weaknesses of each method. Some methods might be more conservative (high precision, low recall), while others might be more liberal in identifying causal links (high recall, low precision).
# 
# From this analysis, we can draw initial conclusions about the relative strengths and robustness of methods like VarLiNGAM, PCMCI, and Dynotears on this collection of datasets. For instance, some methods may show a high median F1 score but also a wide interquartile range, suggesting inconsistent performance. In contrast, another method might have a slightly lower median but be far more stable across different scenarios.
# 
# It's crucial to remember that these results are based on the specific data generation processes included in this benchmark. Further analysis on a wider variety of more complex and realistic datasets is necessary for more generalizable conclusions.

AttributeError: 'list' object has no attribute 'items'